# Modelling an exchange-traded futures contract and handling instrument lifecycle events

This notebook demonstrates the concepts described in the following KBs:

1. Mastering a `Future` instrument, establishing a position, and performing a valuation: https://support.lusid.com/docs/modelling-exchange-traded-futures-and-options-in-lusid
2. Handling mark-to market and expiry instrument lifecyle events: https://support.lusid.com/docs/handling-future-mark-to-market-and-expiry-events

Instrument:
* Start date: 21 September 2024
* Maturity date: 21 March 2025 5:30pm
* Delivery type: Cash
* Contract size: 50
* Contracts: 1

Buy transaction:
* Transaction/settlement date: Wed 25 September 2024 9am
* Units: 10
* Market price: 6000
* Fees (set as property): 50
* Notional amount bought = 10 * 6000 * 1 * 50 = 3,000,000 (units * price * contracts * contract size)

Sell transaction (half the units a week into the contract):
* Transaction/settlement date: Tues 1 October 2024 9am
* Units 5
* Market price: 6001
* Fees (set as property): 50
* Notional amount sold = 5 * 6002 * 1 * 50 = 1,500,500

## Setup

In [2]:
 # Set up LUSID
import os
import pandas as pd
import json
import uuid
from IPython.core.display import HTML
import logging
from datetime import datetime, timezone, timedelta
logging.basicConfig(level = logging.INFO)

import finbourne.sdk.services.lusid.api as la
import finbourne.sdk.services.lusid.models as lm

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.lpt.lpt import to_date

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)
    
# Confirm success by printing SDK version
api_status = pd.DataFrame(api_factory.build(la.ApplicationMetadataApi).get_lusid_versions().to_dict())
display(api_status)

,apiVersion,buildVersion,excelVersion,links
0,v0,0.6.15952.0,0.5.3666,"{'relation': 'RequestLogs', 'href': 'https://j..."


In [29]:
# Build all the required APIs
try:
    instruments_api = api_factory.build(la.InstrumentsApi)
    instrument_events_api = api_factory.build(la.InstrumentEventsApi)
    instrument_event_type_api = api_factory.build(la.InstrumentEventTypesApi)
    aggregation_api = api_factory.build(la.AggregationApi)
    recipe_api = api_factory.build(la.ConfigurationRecipeApi)
    quotes_api = api_factory.build(la.QuotesApi)
    property_definition_api = api_factory.build(la.PropertyDefinitionsApi)
    transaction_portfolios_api = api_factory.build(la.TransactionPortfoliosApi)
    portfolios_api = api_factory.build(la.PortfoliosApi)
    transaction_config_api = api_factory.build(la.TransactionConfigurationApi)
    print("All APIs built correctly")
except ApiException as e:
    print(e)

All APIs built correctly


## Create a scope and code for entities in the Notebook

Keeps data segregated from other data in LUSID.

In [30]:
module_scope = "FBNTutorials"
module_code = "FuturesTest4"
print(f"'{module_scope}\\{module_code}' scope and code created.")

'FBNTutorials\FuturesTest4' scope and code created.


## Create a property type to represent transaction fees

In [31]:
def create_property_type(property_domain, property_scope, property_code, data_type):
    property_type_request = lm.CreatePropertyDefinitionRequest(
        domain = property_domain,
        scope = property_scope,
        code = property_code,
        display_name = property_code,
        data_type_id = lm.ResourceId(scope = "system", code = data_type)
    )

    try:
        property_type_response = property_definition_api.create_property_definition(
            create_property_definition_request = property_type_request
        )
        print(f"Property type created with the following key: {property_type_response.key}")
        return property_type_response.key
    except ApiException as e:
        if json.loads(e.body)["name"] == "PropertyAlreadyExists":
            logging.info(
                f"Property type with the following key already exists: {property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"
            )  
        return f"{property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"

In [32]:
txn_fee_property_key = create_property_type("Transaction", f"{module_scope}{module_code}", "TotalCapitalisedFees", "number")

INFO:root:Property type with the following key already exists: Transaction/FBNTutorialsFuturesTest4/TotalCapitalisedFees


## Create transaction types and sides

Required for both buy/sell transactions and for transactions automatically generated by instrument events.

All created in a custom scope, which must be registered with the transaction portfolio in which transactions are loaded.

In [33]:
def check_TT(tt, scope):
    try:
        tt_response = transaction_config_api.get_transaction_type(source = f"default", type = tt, scope=scope)
        print(f"\n{tt} transaction type:")
        display(lusid_response_to_data_frame(tt_response.aliases))
        display(lusid_response_to_data_frame(tt_response.movements))
        display(lusid_response_to_data_frame(tt_response.calculations))
    except ApiException as e:
        print(e)
        
def check_side(side, scope):
    try:
        side_response = transaction_config_api.get_side_definition(scope = scope, side = side)
        print(f"\n{side} side:")
        side_response_df = lusid_response_to_data_frame(side_response).transpose()
        side_response_df.drop(side_response_df.filter(regex='links').columns, axis=1, inplace=True)
        display(side_response_df)  
    except ApiException as e:
        print(e)

### Create sides

Must be created before transaction types. Includes recreating the built-in `Side1` and `Side2` in the custom scope.

In [34]:
# Recreate Side1
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TradeAmount"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    if json.loads(e.body)["name"] == "InvalidParameterValue":
        logging.info("Side definition already exists.")

Success


In [35]:
# Recreate Side2
side_definition = lm.SideDefinitionRequest(
    security = "Txn:SettleCcy",
    currency = "Txn:SettlementCurrency",
    rate = "SettledToPortfolioRate",
    units = "Txn:TotalConsideration",
    amount = "Txn:TotalConsideration"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side2",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    if json.loads(e.body)["name"] == "InvalidParameterValue":
        logging.info("Side definition already exists.")

Success


In [36]:
# Create custom 'Notional' side to record the notional amount
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TotalConsideration", 
    notionalAmount="Transaction/default/NotionalAmount"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Notional",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    if json.loads(e.body)["name"] == "InvalidParameterValue":
        logging.info("Side definition already exists.")

Success


In [37]:
# Create custom 'MTM' side for mark-to-market event
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TotalConsideration"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "MTM",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    if json.loads(e.body)["name"] == "InvalidParameterValue":
        logging.info("Side definition already exists.")

Success


### Create transaction types

#### Create `BuyFuture` and `SellFuture` transaction types (to establish positions)

Note the transaction type calculations are mandatory to calculate the correct gross consideration and notional amount. See https://support.lusid.com/docs/modelling-exchange-traded-futures-and-options-in-lusid#booking-a-transaction-to-establish-a-position.

Both `StockMovement` use the custom `Notional` side.

In [38]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "BuyFuture",
            description = "Open the contract",
            transaction_class = "Futures",
            transaction_roles = "LongLonger",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            side = "Notional",
            direction = 1
        ),
        lm.TransactionTypeMovement(
            movement_types = "CashCommitment",
            side = "Side2",
            direction = -1
        )
    ],
    calculations = [
        lm.TransactionTypeCalculation(
            type = "Txn:NotionalAmount"
        ),
        lm.TransactionTypeCalculation(
            type = "Txn:GrossConsideration"
        ),
        # Total consideration is gross plus fees
        lm.TransactionTypeCalculation(
            type = "DeriveTotalConsideration",
            formula = f"Txn:GrossConsideration + Properties[{txn_fee_property_key}]"
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        source = "default",
        scope = f"{module_scope}{module_code}",
        # The primary alias name
        type = "BuyFuture",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [39]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "SellFuture",
            description = "Close the contract",
            transaction_class = "Futures",
            transaction_roles = "LongShorter",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            side = "Notional",
            direction = -1
        ),
        lm.TransactionTypeMovement(
            movement_types = "CashCommitment",
            side = "Side2",
            direction = 1
        )
    ],
    calculations = [
        lm.TransactionTypeCalculation(
            type = "Txn:NotionalAmount"
        ),
        lm.TransactionTypeCalculation(
            type = "Txn:GrossConsideration"
        ),
        # Total consideration is gross minus fees
        lm.TransactionTypeCalculation(
            type = "DeriveTotalConsideration",
            formula = f"Txn:GrossConsideration - Properties[{txn_fee_property_key}]"
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        source = "default",
        scope = f"{module_scope}{module_code}",
        # The primary alias name
        type = "SellFuture",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create `FutureMarkToMarket` transaction type (to handle `FutureMarkToMarketEvent`)

Note the `DeriveTotalConsideration` calculation is mandatory, because the event sets total consideration to 0 out-of-the-box. See https://support.lusid.com/docs/handling-future-mark-to-market-and-expiry-events#creating-a-suitable-futuremarktomarket-transaction-type

The `VariationMargin` movement uses the custom `MTM` side. 

In [40]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "FutureMarkToMarket",
            description = "Increase cost basis and adjust cash",
            transaction_class = "Futures",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "VariationMargin",
            side = "MTM",
            direction = 1
        ),
        lm.TransactionTypeMovement(
            movement_types = "CashReceivable",
            side = "Side2",
            direction = 1
        )
    ],
    calculations = [
        lm.TransactionTypeCalculation(
            type = "DeriveTotalConsideration",
            formula = "Txn:GrossConsideration"
        )
    ]
)
try:
    response = transaction_config_api.set_transaction_type(
        # The source must be `default` to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # The primary alias name
        type = "FutureMarkToMarket",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create `FutureCashSettlement` transaction type (to handle `FutureExpiryEvent`)

Note the transaction type calculations are mandatory. See https://support.lusid.com/docs/handling-future-mark-to-market-and-expiry-events#creating-a-suitable-futurecashsettlement-transaction-type.

The `StockMovement` uses the custom `Notional` side. 

In [41]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "FutureCashSettlement",
            description = "Cash settle at expiry",
            transaction_class = "Futures",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            side = "Notional",
            direction = -1
        ),
        lm.TransactionTypeMovement(
            movement_types = "CashReceivable",
            side = "Side2",
            direction = 1
        )
    ],
    calculations = [
        lm.TransactionTypeCalculation(
            type = "Txn:GrossConsideration"
        ),
        lm.TransactionTypeCalculation(
            type = "DeriveTotalConsideration",
            formula = "Txn:GrossConsideration"
        )
    ]
)
try:
    response = transaction_config_api.set_transaction_type(
        # The source must be `default` to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "FutureCashSettlement",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [42]:
check_TT("BuyFuture", f"{module_scope}{module_code}")
check_TT("SellFuture", f"{module_scope}{module_code}")
check_TT("FutureMarkToMarket", f"{module_scope}{module_code}")
check_TT("FutureCashSettlement", f"{module_scope}{module_code}")
check_side("Side1", f"{module_scope}{module_code}")
check_side("Side2", f"{module_scope}{module_code}")
check_side("Notional", f"{module_scope}{module_code}")
check_side("MTM", f"{module_scope}{module_code}")


BuyFuture transaction type:


,type,description,transaction_class,transaction_roles,is_default
0,BuyFuture,Open the contract,Futures,LongLonger,False


,movement_types,side,direction,properties,mappings,movement_options,condition,settlement_mode
0,StockMovement,Notional,1,{},[],[],,Internal
1,CashCommitment,Side2,-1,{},[],[],,Internal


,type,formula
0,Txn:NotionalAmount,None
1,Txn:GrossConsideration,None
2,DeriveTotalConsideration,Txn:GrossConsideration + Properties[Transactio...



SellFuture transaction type:


...

,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Side2,Txn:SettleCcy,Txn:SettlementCurrency,SettledToPortfolioRate,Txn:TotalConsideration,Txn:TotalConsideration,0,None



Notional side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Notional,Txn:LusidInstrumentId,Txn:TradeCurrency,Txn:TradeToPortfolioRate,Txn:Units,Txn:TotalConsideration,Transaction/default/NotionalAmount,None



MTM side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,MTM,Txn:LusidInstrumentId,Txn:TradeCurrency,Txn:TradeToPortfolioRate,Txn:Units,Txn:TotalConsideration,0,None


## Master a `Future` instrument

Note the following:

* There's no explicit underlying (ie. no `MasteredInstrument`).
* `deliveryType` is `Physical` by default but must be set to `Cash` to trigger instrument events.
* `contracts` is nominally an optional field but is mandatory for instrument events and should be set to `1`.
* `markToMarketConventions` is mandatory if you want LUSID to emit a daily `FutureMarkToMarketEvent`.
* The time component of `maturityDate` must match the time settlement prices are released by the exchange, in order to feed market data to `FutureMarkToMarketEvent`.

For more information, see https://support.lusid.com/docs/handling-future-mark-to-market-and-expiry-events#mastering-an-instrument-and-establishing-a-position.

In [49]:
def master_instrument(id, currency, start, end):
    
    instrument_request = {
        id: lm.InstrumentDefinition(
            name = id,
            identifiers = {"ClientInternal": lm.InstrumentIdValue(value = id)},
            definition = lm.Future(
                instrument_type = "Future", 
                start_date = datetime.strptime(start, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
                maturity_date = datetime.strptime(end, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
                identifiers = {},
                contractDetails=lm.FuturesContractDetails(
                    domCcy=currency,
                    contractCode="ESHS",
                    contractMonth="H",
                    contractSize = 50,
                    exchangeCode="GLOBEX",
                    deliveryType="Cash"
                ),
                contracts=1,
                markToMarketConventions = lm.MarkToMarketConventions(
                    calendarCode=currency
                ),
                tradingConventions=lm.TradingConventions(
                    priceScaleFactor=1
                )
            )
        )
    }
    
    try:
        instrument_response = instruments_api.upsert_instruments(
            request_body = instrument_request,
            scope = f"{module_scope}{module_code}"
        )
        # Return LUID from (only) Instrument object
        return list(instrument_response.values.values())[0].lusid_instrument_id
    except ApiException as e:
        print(e)

In [50]:
# Master forward in custom instrument scope and capture LUID
luid_dict = {}
luid_dict["EminiS&P500FuturesContract"] = master_instrument("EminiS&P500FuturesContract", "USD", "2024-09-21 00:00:00", "2025-03-21 17:30:00")

for k, v in luid_dict.items():
    print(f"{k}: {v}")

EminiS&P500FuturesContract: LUID_00003H17


In [51]:
def list_instrs():
    instr_response = instruments_api.list_instruments(scope=f"{module_scope}{module_code}")
    instr_response_df = lusid_response_to_data_frame(instr_response)
    instr_response_df.drop(instr_response_df.filter(regex='version|href|staged').columns, axis=1, inplace=True)
    display(instr_response_df.transpose())

list_instrs()

,0
scope,FBNTutorialsFuturesTest4
lusid_instrument_id,LUID_00003H17
name,EminiS&P500FuturesContract
identifiers.ClientInternal,EminiS&P500FuturesContract
identifiers.LusidInstrumentId,LUID_00003H17
properties,[]
instrument_definition.instrument_type,Future
state,Active
asset_class,Unknown
dom_ccy,USD


## Create CTVoM recipe

Must be specified as a portfolio recipe to enable instrument events. Can also be used as a valuation recipe.

Note CTVoM is the recommended (though not the default) pricing model for `Future` instruments. It sets PV to the unrealised gain/loss whether you choose to realise it daily or not.

In [52]:
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as the portfolio
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-CTVoM",
    description = "A recipe to value a future",
    market = lm.MarketContext(
        market_rules = [
            # Look up FX spot rates in the LUSID quote store, if needed
            lm.MarketDataKeyRule(
                key = "Fx.CurrencyPair.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Rate",
                field = "mid",
                quote_interval = "0D.0D",
            ),
            # Look up price of future contract (not underlying)
            lm.MarketDataKeyRule(
                key = "Quote.LusidInstrumentId.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Price",
                field = "mid",
                quote_interval = "0D.0D",
            )
        ]
    ),
    # Change pricing model
    pricing=lm.PricingContext(
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                instrument_type="Future",
                model_name="ConstantTimeValueOfMoney"
            )
        ]
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [53]:
# Confirm upsert and show the many options that are automatically set to default values by LUSID.
config_recipe = recipe_api.list_configuration_recipes(filter=f"value.scope eq '{module_scope}' and value.code startswith '{module_code}'")
config_recipe_df = lusid_response_to_data_frame(config_recipe)
display(config_recipe_df.transpose())

,0
value.scope,FBNTutorials
value.code,FuturesTest4-CTVoM
value.market.market_rules.0.key,Fx.CurrencyPair.*
value.market.market_rules.0.supplier,Lusid
value.market.market_rules.0.data_scope,FBNTutorialsFuturesTest4
value.market.market_rules.0.quote_type,Rate
value.market.market_rules.0.var_field,mid
value.market.market_rules.0.quote_interval,0D.0D
value.market.market_rules.0.price_source,
value.market.market_rules.0.source_system,Lusid


## Set up a USD transaction portfolio

With the portfolio recipe set to the CTVoM recipe created above to enable instrument events, and the transaction type scope registered.

In [56]:
portfolio_request=lm.CreateTransactionPortfolioRequest(
    display_name = f"Future portfolio",
    code = f"{module_code}",
    # Set the portfolio currency
    base_currency = "USD",
    # Must be before first transaction recorded
    created = datetime.strptime("2024-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
    # Attempt to resolve transactions to instruments in the custom scope before falling back to the default scope
    instrument_scopes = [f"{module_scope}{module_code}"],
    # Register transaction type scope
    transactionTypeScope=f"{module_scope}{module_code}",
    instrumentEventConfiguration=lm.InstrumentEventConfiguration(
        recipeId=lm.ResourceId(
            scope=module_scope,
            code=f"{module_code}-CTVoM"
        )
    )
)

try:
    portfolio_response=transaction_portfolios_api.create_portfolio(
        scope = module_scope,
        create_transaction_portfolio_request = portfolio_request
    )
    print(f"Portfolio with display name '{portfolio_response.display_name}' created effective {str(portfolio_response.created)}")
except ApiException as e:
    print(e)

Portfolio with display name 'Future portfolio' created effective 2024-01-01 00:00:00+00:00


### Confirm portfolio details

In [57]:
def get_port_details():
    portfolio_response = transaction_portfolios_api.get_details(scope = module_scope, code = f"{module_code}")
    portfolio_response_df = lusid_response_to_data_frame(portfolio_response).transpose()
    # Drop some noisy columns
    portfolio_response_df.drop(portfolio_response_df.filter(regex='version|href|staged|links|settlement').columns, axis=1, inplace=True)
    display(portfolio_response_df.transpose())
    
get_port_details()

,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,FuturesTest4
base_currency,USD
corporate_action_source_id,None
sub_holding_keys,[]
instrument_scopes.0,FBNTutorialsFuturesTest4
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsFuturesTest4
cash_gain_loss_calculation_date,Default


## Establish a position

Note the following:

* LUSID calculates gross consideration as `0` for transactions that increase a position in a `Future` instrument.
* `totalConsideration.amount` must be set to `0` to trigger LUSID to calculate total consideration as gross + fees.

For more information, see https://support.lusid.com/docs/modelling-exchange-traded-futures-and-options-in-lusid#booking-a-transaction-to-establish-a-position.

In [60]:
def create_transactions(txnid, tttype, luid, date, quantity, price, ccy, fees):
    
    create_txn_request = {
        txnid: lm.TransactionRequest(
            transaction_id=txnid,
            type=tttype,
            instrument_identifiers = {"Instrument/default/LusidInstrumentId": luid},
            transaction_date=datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            settlement_date=datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            units=quantity,
            # This is the market price, used for gross consideration calculation
            transaction_price=lm.TransactionPrice(
                price=price, type="Price"
            ),
            
            total_consideration = lm.CurrencyAndAmount(
                currency = ccy,
                amount = 0
            ),
            properties={
                txn_fee_property_key: lm.PerpetualProperty(
                    key = txn_fee_property_key,
                    value = lm.PropertyValue(
                        metric_value = lm.MetricValue(
                            value = fees,
                        )
                    )
                ),
            }
        )
    }
    
    try:
        create_txn_response = transaction_portfolios_api.batch_upsert_transactions(
            scope = f"{module_scope}",
            code = f"{module_code}",
            success_mode="Partial",
            request_body = create_txn_request
        )
        print(create_txn_response.failed) if create_txn_response.failed else print("Success")
    except ApiException as e:
        print(e)

In [61]:
create_transactions("Txn01", "BuyFuture", luid_dict["EminiS&P500FuturesContract"], "2024-09-25 09:00:00", 10, 6000, "USD", 50)

Success


### Confirm positions and audit output transactions

In [67]:
def get_portfolio_holdings(date):           
    if date == "today":
        date = str(datetime.datetime.now().replace(microsecond=0))

    try:
        get_holdings_response = transaction_portfolios_api.get_holdings(
            scope = module_scope, 
            code = f"{module_code}",
            effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
        )
        get_holdings_response_df = lusid_response_to_data_frame(get_holdings_response)
        # Drop some noisy columns
        get_holdings_response_df.drop(get_holdings_response_df.filter(regex='properties').columns, axis=1, inplace=True)
        display(get_holdings_response_df)
    except ApiException as e:
        print(e)

In [68]:
def get_output_transactions(start, end):

    try:
        output_transactions_response = transaction_portfolios_api.build_transactions(
            scope = module_scope, 
            code = f"{module_code}",
            transaction_query_parameters = lm.TransactionQueryParameters(
                start_date = datetime.strptime(start, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                end_date = datetime.strptime(end, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat()
            )
        )
        output_transactions_response_df = lusid_response_to_data_frame(output_transactions_response)
        display(output_transactions_response_df.transpose())
    except ApiException as e:
        print(e)

In [69]:
get_portfolio_holdings("2024-09-25 09:00:00")   # Trade/settlement datetime
get_output_transactions("2024-01-01", "2030-01-01")

,instrument_scope,instrument_uid,sub_holding_keys,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFuturesTest4,LUID_00003H17,{},P,10.00,10.00,50.00,USD,50.00,USD,USD,Position,80357266,"3,000,000.00",USD,50.00,USD,50.00,USD,0.00,USD,0.00,USD,[],0.00,0.00
1,default,CCY_USD,{},B,-50.00,-50.00,-50.00,USD,-50.00,USD,USD,Balance,80357267,0.00,USD,-50.00,USD,-50.00,USD,0.00,USD,0.00,USD,[],0.00,0.00


,0
transaction_id,Txn01
type,BuyFuture
description,Open the contract
instrument_identifiers.Instrument/default/LusidInstrumentId,LUID_00003H17
instrument_scope,FBNTutorialsFuturesTest4
instrument_uid,LUID_00003H17
transaction_date,2024-09-25 09:00:00+00:00
settlement_date,2024-09-25 09:00:00+00:00
units,10.00
transaction_amount,0.00


### Sell half the contract a week in

LUSID calculates gross consideration as `notionalAmount - (notionalCost + variationMargin)` for transactions that decrease a position in a `Future` instrument. See https://support.lusid.com/docs/what-is-a-transaction-type-calculation#txngrossconsideration-calculation.

In [70]:
create_transactions("Txn02", "SellFuture", luid_dict["EminiS&P500FuturesContract"], "2024-10-01 09:00:00", 5, 6002, "USD", 50)

Success


In [71]:
get_portfolio_holdings("2024-10-01 09:00:00")   # Trade/settlement datetime
get_output_transactions("2024-01-01", "2030-01-01")

,instrument_scope,instrument_uid,sub_holding_keys,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFuturesTest4,LUID_00003H17,{},P,5.00,5.00,25.00,USD,25.00,USD,USD,Position,80357266,"1,500,000.00",USD,25.00,USD,25.00,USD,0.00,USD,0.00,USD,[],0.00,0.00
1,default,CCY_USD,{},B,400.00,400.00,400.00,USD,400.00,USD,USD,Balance,80357267,0.00,USD,400.00,USD,400.00,USD,0.00,USD,0.00,USD,[],0.00,0.00


,0,1
transaction_id,Txn01,Txn02
type,BuyFuture,SellFuture
description,Open the contract,Close the contract
instrument_identifiers.Instrument/default/LusidInstrumentId,LUID_00003H17,LUID_00003H17
instrument_scope,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4
instrument_uid,LUID_00003H17,LUID_00003H17
transaction_date,2024-09-25 09:00:00+00:00,2024-10-01 09:00:00+00:00
settlement_date,2024-09-25 09:00:00+00:00,2024-10-01 09:00:00+00:00
units,10.00,5.00
transaction_amount,0.00,0.00


## Load market prices

Market prices are required both for instrument events and for any date on which you want to perform an intra-day valuation. See https://support.lusid.com/docs/handling-future-mark-to-market-and-expiry-events#loading-suitable-market-data-into-the-lusid-quote-store.

All prices for events are loaded at 5:30pm, as per the time specified in the instrument `maturityDate`.

Multiple prices are loaded for the purchase date of Wed 25 September 2024 5:30pm and for a week afterwards, to demonstrate `FutureMarkToMarketEvent` (which is only emitted on weekdays, not weekends, up until the day **before** the maturity date).

One price is loaded for the maturity date: Friday 21 March 2025 5:30pm, to demonstrate `FutureExpiryEvent`.

One price is loaded for a random intra-day valuation: Thursday 26 September 2024 12pm.

In [73]:
# Create convenience function
def load_quotes(luid, price, date, ccy, scale):
    if date == "today":
        date = str(datetime.datetime.now().replace(microsecond=0))

    quotes = {
        # Each quote must be upserted with an ephemeral key (uuid in this case), to track errors in the response
        str(uuid.uuid4()): lm.UpsertQuoteRequest(
            quote_id = lm.QuoteId(
                quote_series_id = lm.QuoteSeriesId(
                    # Must be one of the valid financial data vendor 'provider' values
                    provider = "Lusid",
                    instrument_id_type = "LusidInstrumentId",
                    instrument_id = luid,
                    quote_type = "Price",
                    # Case sensitive: the field value must match that of the equivalent recipe field exactly
                    field = "mid",
                ),
                effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            ),
            metric_value = lm.MetricValue(value = price, unit = ccy),
            scale_factor = scale,
        )
    }

    try:
        upsert_quotes_response = quotes_api.upsert_quotes(scope = f"{module_scope}{module_code}", request_body = quotes)    
        if upsert_quotes_response.failed == {}:
            print(f"Price for {date} successfully loaded into LUSID.")
        else:
            print(f"Some failures occurred. {len(upsert_quotes_response.failed)} prices did not get loaded into LUSID.")
    except ApiException as e:
        print(e)

In [74]:
# First week or so after initial purchase
load_quotes(luid_dict["EminiS&P500FuturesContract"], 6000, "2024-09-25 09:00:00", "USD", 1) # Wed - Price at SOD (purchase date/time)
load_quotes(luid_dict["EminiS&P500FuturesContract"], 6002, "2024-09-25 17:30:00", "USD", 1) # Wed - Price at EOD for MTM event
load_quotes(luid_dict["EminiS&P500FuturesContract"], 6003, "2024-09-26 12:00:00", "USD", 1) # Thur intra-day valuation 
load_quotes(luid_dict["EminiS&P500FuturesContract"], 6004, "2024-09-26 17:30:00", "USD", 1) # Thur EOD for MTM event
load_quotes(luid_dict["EminiS&P500FuturesContract"], 5998, "2024-09-27 17:30:00", "USD", 1) # Fri EOD for MTM event
load_quotes(luid_dict["EminiS&P500FuturesContract"], 5998, "2024-09-28 17:30:00", "USD", 1) # Sat - no MTM event
load_quotes(luid_dict["EminiS&P500FuturesContract"], 5998, "2024-09-29 17:30:00", "USD", 1) # Sun - no MTM event
load_quotes(luid_dict["EminiS&P500FuturesContract"], 6001, "2024-09-30 17:30:00", "USD", 1) # Mon EOD for MTM event
load_quotes(luid_dict["EminiS&P500FuturesContract"], 6003, "2024-10-01 17:30:00", "USD", 1) # Tues EOD for MTM event, after sale of half the contract

# Maturity date
load_quotes(luid_dict["EminiS&P500FuturesContract"], 6005, "2025-03-21 17:30:00", "USD", 1) # Price at EOD for Expiry event

Price for 2024-09-25 09:00:00 successfully loaded into LUSID.
Price for 2024-09-25 17:30:00 successfully loaded into LUSID.
Price for 2024-09-26 12:00:00 successfully loaded into LUSID.
Price for 2024-09-26 17:30:00 successfully loaded into LUSID.
Price for 2024-09-27 17:30:00 successfully loaded into LUSID.
Price for 2024-09-28 17:30:00 successfully loaded into LUSID.
Price for 2024-09-29 17:30:00 successfully loaded into LUSID.
Price for 2024-09-30 17:30:00 successfully loaded into LUSID.
Price for 2024-10-01 17:30:00 successfully loaded into LUSID.
Price for 2025-03-21 17:30:00 successfully loaded into LUSID.


In [75]:
def list_quotes():
    try:
        quotes_response = quotes_api.list_quotes_for_scope(f"{module_scope}{module_code}")
        quotes_response_df = lusid_response_to_data_frame(quotes_response)
        display(quotes_response_df)        
    except ApiException as e:
        print(e)

list_quotes()

,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2025-03-21T17:30:00.0000000+00:00,"6,005.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:07.004757+00:00,1.00
1,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-10-01T17:30:00.0000000+00:00,"6,003.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:06.848850+00:00,1.00
2,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-30T17:30:00.0000000+00:00,"6,001.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:06.692991+00:00,1.00
3,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-29T17:30:00.0000000+00:00,"5,998.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:06.196419+00:00,1.00
4,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-28T17:30:00.0000000+00:00,"5,998.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:05.523714+00:00,1.00
5,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-27T17:30:00.0000000+00:00,"5,998.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:05.373725+00:00,1.00
6,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-26T17:30:00.0000000+00:00,"6,004.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:05.229664+00:00,1.00
7,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-26T12:00:00.0000000+00:00,"6,003.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:05.024202+00:00,1.00
8,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-25T17:30:00.0000000+00:00,"6,002.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:04.822929+00:00,1.00
9,Lusid,LUID_00003H17,LusidInstrumentId,Price,mid,a1ac33be-299f-4469-8166-2625ffd49bbc,2024-09-25T09:00:00.0000000+00:00,"6,000.00",USD,,,00u91lo2d7X42sdse2p7,2026-06-05 08:05:04.421193+00:00,1.00


## Value the portfolio and understand impact of instrument events

In [76]:
def value_instruments(date, pnl_window):
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))

    valuation_request = lm.ValuationRequest(
        # Choose recipe to use
        recipe_id = lm.ResourceId(scope = module_scope, code = f"{module_code}-CTVoM"),
        # Specify metrics (also known as queryable keys) to report useful information
        metrics = [
            lm.AggregateSpec(key="Instrument/InstrumentCategory", op="Value"),
#            lm.AggregateSpec(key="Valuation/Model/Name", op="Value"),
            lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
            lm.AggregateSpec(key="Instrument/default/Name", op="Value"),
            lm.AggregateSpec(key="Valuation/EffectiveAt", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Value"),
            lm.AggregateSpec(key="Quotes/PriceOrFXRate", op="Value"),
            lm.AggregateSpec(key="Holding/Cost/Dom", op="Value"),
#            lm.AggregateSpec(key="Valuation/CleanPV", op="Value"),
            lm.AggregateSpec(key="Valuation/PV", op="Value"),
#            lm.AggregateSpec(key="Valuation/Accrued", op="Value"),
            lm.AggregateSpec(key="Valuation/Exposure", op="Value"),
#            lm.AggregateSpec(key="Valuation/CurrentNotional", op="Value"), 
            lm.AggregateSpec(key="ProfitAndLoss/Total", op="Value", options={"Window": f"{pnl_window}"}),      
            lm.AggregateSpec(key="ProfitAndLoss/Total/Market", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="ProfitAndLoss/Realised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Unrealised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Total/Other", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="Aggregation/Errors", op="Value"), 
        ],
        # Identify portfolio to value
        portfolio_entity_ids = [lm.PortfolioEntityId(scope = module_scope, code = f"{module_code}")],
        valuation_schedule = lm.ValuationSchedule(effective_at = date),

    )

    try:
        # Get portfolio valuation
        val_response = aggregation_api.get_valuation(valuation_request = valuation_request)
        val_response_df = pd.json_normalize(val_response.to_dict()["data"], sep='.')
        # Rename columns
        val_response_df.rename(
            columns = {
                "Instrument/InstrumentCategory": "Category",
                "Valuation/Model/Name": "Model",
                "Instrument/default/LusidInstrumentId": "LUID",
                "Instrument/default/Name": "Name",
                "Valuation/EffectiveAt": "Date",
                "Holding/default/Units": "Units",
                "Quotes/PriceOrFXRate": "Price",
                "Quotes/ScaleFactor": "Quote Scale Factor",
                "Holding/Cost/Dom": "Local Cost",
                "Valuation/CleanPV": "Local Clean PV",
                "Valuation/PV": "Local PV",
                "Valuation/Accrued": "Local Accrued Interest",
                "Valuation/Exposure": "Exposure",
                "Valuation/CurrentNotional": "Notional",            
                f"ProfitAndLoss/Total(Window=\"{pnl_window}\")": "Total P&L",
                f"ProfitAndLoss/Total/Market(Window=\"{pnl_window}\")": "Total/Market P&L",
                f"ProfitAndLoss/Realised/Market(Window=\"{pnl_window}\")": "Realised/Market P&L",
                f"ProfitAndLoss/Unrealised/Market(Window=\"{pnl_window}\")": "Unrealised/Market P&L",
                f"ProfitAndLoss/Total/Other(Window=\"{pnl_window}\")": "Total/Other P&L",
                "Aggregation/Errors": "Errors"
            },
            inplace = True,
        )
    #     #val_each_instrument_df["Date"] = pd.to_datetime(val_each_instrument_df["Date"]).dt.date
        display(val_response_df)
    except ApiException as e:
        print(e)

In [77]:
# First week or so after initial purchase
value_instruments("2024-09-25T09:01:00Z", "Daily") # Wed SOD - Just after transaction datetime
value_instruments("2024-09-25T17:31:00Z", "Daily") # Wed EOD - After first MTM event
value_instruments("2024-09-26T12:01:00Z", "Daily") # Thur intra-day valuation
value_instruments("2024-09-26T17:31:00Z", "Daily") # Thur EOD
value_instruments("2024-09-27T17:31:00Z", "Daily") # Fri EOD
value_instruments("2024-09-28T17:31:00Z", "Daily") # Sat EOD - No MTM event
value_instruments("2024-09-29T17:31:00Z", "Daily") # Sun EOD - No MTM event
value_instruments("2024-09-30T17:31:00Z", "Daily") # Mon EOD
value_instruments("2024-10-01T17:31:00Z", "Daily") # Tues EOD, after sale of half the contract

# Maturity
value_instruments("2025-05-21T17:31:00Z", "Daily")

,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-25T09:01:00.0000000+00:00,10.00,"6,000.00",50.00,0.00,"3,000,000.00",-50.00,-50.00,0.00,-50.00,0.00,[]
1,Cash,CCY_USD,USD,2024-09-25T09:01:00.0000000+00:00,-50.00,1.00,-50.00,-50.00,-50.00,0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-25T17:31:00.0000000+00:00,10.00,"6,002.00",50.00,0.00,"3,001,000.00","-1,050.00","-1,050.00",0.00,"-1,050.00",0.00,[]
1,Cash,CCY_USD,USD,2024-09-25T17:31:00.0000000+00:00,950.00,1.00,950.00,950.00,950.00,0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-26T12:01:00.0000000+00:00,10.00,"6,003.00",50.00,500.00,"3,001,500.00",500.00,500.00,0.00,500.00,0.00,[]
1,Cash,CCY_USD,USD,2024-09-26T12:01:00.0000000+00:00,950.00,1.00,950.00,950.00,950.00,0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-26T17:31:00.0000000+00:00,10.00,"6,004.00",50.00,0.00,"3,002,000.00","-1,000.00","-1,000.00",0.00,"-1,000.00",0.00,[]
1,Cash,CCY_USD,USD,2024-09-26T17:31:00.0000000+00:00,"1,950.00",1.00,"1,950.00","1,950.00","1,950.00",0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-27T17:31:00.0000000+00:00,10.00,"5,998.00",50.00,0.00,"2,999,000.00","3,000.00","3,000.00",0.00,"3,000.00",0.00,[]
1,Cash,CCY_USD,USD,2024-09-27T17:31:00.0000000+00:00,"-1,050.00",1.00,"-1,050.00","-1,050.00","-1,050.00",0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-28T17:31:00.0000000+00:00,10.00,"5,998.00",50.00,0.00,"2,999,000.00",0.00,0.00,0.00,0.00,0.00,[]
1,Cash,CCY_USD,USD,2024-09-28T17:31:00.0000000+00:00,"-1,050.00",1.00,"-1,050.00","-1,050.00","-1,050.00",0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-29T17:31:00.0000000+00:00,10.00,"5,998.00",50.00,0.00,"2,999,000.00",0.00,0.00,0.00,0.00,0.00,[]
1,Cash,CCY_USD,USD,2024-09-29T17:31:00.0000000+00:00,"-1,050.00",1.00,"-1,050.00","-1,050.00","-1,050.00",0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-09-30T17:31:00.0000000+00:00,10.00,"6,001.00",50.00,0.00,"3,000,500.00","-1,500.00","-1,500.00",0.00,"-1,500.00",0.00,[]
1,Cash,CCY_USD,USD,2024-09-30T17:31:00.0000000+00:00,450.00,1.00,450.00,450.00,450.00,0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Future,LUID_00003H17,EminiS&P500FuturesContract,2024-10-01T17:31:00.0000000+00:00,5.00,"6,003.00",25.00,0.00,"1,500,750.00",200.00,200.00,425.00,-225.00,0.00,[]
1,Cash,CCY_USD,USD,2024-10-01T17:31:00.0000000+00:00,"1,150.00",1.00,"1,150.00","1,150.00","1,150.00",0.00,0.00,0.00,0.00,0.00,[]


,Category,LUID,Name,Date,Units,Price,Local Cost,Local PV,Exposure,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Cash,CCY_USD,USD,2025-05-21T17:31:00.0000000+00:00,"1,650.00",1.00,"1,650.00","1,650.00","1,650.00",0.00,0.00,0.00,0.00,0.00,[]


### Check holdings after maturity

In [78]:
get_portfolio_holdings("2025-03-21 23:59:59")

,instrument_scope,instrument_uid,sub_holding_keys,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,default,CCY_USD,{},B,"1,650.00","1,650.00","1,650.00",USD,"1,650.00",USD,USD,Balance,80357267,0.00,USD,"1,650.00",USD,"1,650.00",USD,0.00,USD,0.00,USD,[],0.00,0.00


### Get output transactions for entirity of contract

In [79]:
get_output_transactions("2024-01-01", "2030-01-01")

,0,1,2,3,4,5,6,7
transaction_id,Txn01,LUID_00003H17_FutureMarkToMarketEvent_20240925...,LUID_00003H17_FutureMarkToMarketEvent_20240926...,LUID_00003H17_FutureMarkToMarketEvent_20240927...,LUID_00003H17_FutureMarkToMarketEvent_20240930...,Txn02,LUID_00003H17_FutureMarkToMarketEvent_20241001...,LUID_00003H17_FutureExpiryEvent_20250321-80357266
type,BuyFuture,FutureMarkToMarket,FutureMarkToMarket,FutureMarkToMarket,FutureMarkToMarket,SellFuture,FutureMarkToMarket,FutureCashSettlement
description,Open the contract,Increase cost basis and adjust cash,Increase cost basis and adjust cash,Increase cost basis and adjust cash,Increase cost basis and adjust cash,Close the contract,Increase cost basis and adjust cash,Cash settle at expiry
instrument_identifiers.Instrument/default/LusidInstrumentId,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17
instrument_scope,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4,FBNTutorialsFuturesTest4
instrument_uid,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17,LUID_00003H17
transaction_date,2024-09-25 09:00:00+00:00,2024-09-25 17:30:00+00:00,2024-09-26 17:30:00+00:00,2024-09-27 17:30:00+00:00,2024-09-30 17:30:00+00:00,2024-10-01 09:00:00+00:00,2024-10-01 17:30:00+00:00,2025-03-21 17:30:00+00:00
settlement_date,2024-09-25 09:00:00+00:00,2024-09-25 17:30:00+00:00,2024-09-26 17:30:00+00:00,2024-09-27 17:30:00+00:00,2024-09-30 17:30:00+00:00,2024-10-01 09:00:00+00:00,2024-10-01 17:30:00+00:00,2025-03-21 17:30:00+00:00
units,10.00,10.00,10.00,10.00,10.00,5.00,5.00,5.00
transaction_amount,0.00,"1,000.00","1,000.00","3,000.00","1,500.00",0.00,500.00,0.00


## Check instrument events that generated output transactions

LUSID emits a `FutureMarkToMarketEvent` every weekday for the entirity of the contract, until the day **before** the maturity date. However, for those days on which no market data can be found (which in this example is every day after Tues 1 October 2024), no output transaction is generated for that event.

Examine the `generatedEventDiagnostics` object for events starting Wed 2 October 2024 to examine the `MarketDataFailure` error. 

In [80]:
def query_instr_events():
    
    query_id_request = lm.QueryApplicableInstrumentEventsRequest(
        window_start = datetime.strptime("2015-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        window_end = datetime.strptime("2030-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        effective_at = datetime.strptime("2025-03-21", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        portfolio_entity_ids = [
            lm.PortfolioEntityId(
                scope = module_scope,
                code = f"{module_code}",
            )
        ],
        forecasting_recipe_id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-CTVoM"
        )
    )
    
    try:
        query_id_response = instrument_events_api.query_applicable_instrument_events(
            query_applicable_instrument_events_request = query_id_request,
            limit = 200
        )
        display(lusid_response_to_data_frame(query_id_response).transpose())
    except ApiException as e:
        print(e)
        
query_instr_events()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120
portfolio_id.scope,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials
portfolio_id.code,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4,FuturesTest4
holding_id,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266,80357266